# NOTEBOOK FEATURE ENGINEERING

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import chi2_contingency
from itertools import combinations
from scipy.stats import f_oneway

import os
import re

from IPython.display import display, Markdown

import missingno as msno
import sys

from rapidfuzz import process, fuzz

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from itertools import product

import warnings
warnings.filterwarnings('ignore')

In [ ]:
sys.path.append(os.path.abspath("../02_Src"))

from Compas.feature_engineering import one_hot_encoding, compute_weight

In [35]:
df = pd.read_csv(r"..\00_Data\00_Processed\df_eda.csv")

In [ ]:
model_variables_list = [
    'person_id',
    'decile_score',
    'rawscore',
    'v_decile_score',
    'is_recid',
    'is_violent_recid',
    'two_year_recid',
    'sex',
    'race',
    'age',
    'total_priors_count',
    'adult_priors_count',
    'juv_priors_count',
    'c_charge_degree',
    'maritalstatus',
    'juv_fel_count',
    'juv_misd_count',
    'juv_other_count',
    'is_violent',
    'is_drug',
    'is_property',
    'is_fraud',
    'is_weapon',
    'is_traffic',
    'is_sexual',
    'is_public_disorder',
    'is_justice_related',
    'no_charge'
]

df = df[model_variables_list]

In [ ]:
# Small samples -> other 

df['race'] = df['race'].replace({'asian': 'other', 'native american': 'other'})
df['maritalstatus'] = df['maritalstatus'].replace({'married': 'significant other', 'divorced': 'separated', 'widowed': 'other', 'unknown': 'other'})


# Replace by 0 and 1
df['sex'] = df['sex'].replace({'male': 0, 'female': 1})
df['c_charge_degree'] = df['c_charge_degree'].replace({'felony': 0, 'misdemeanor': 1})

In [51]:
df_model = pd.concat([df, one_hot_encoding(df, 'maritalstatus', 'single')], axis = 1)

In [ ]:
# Log to handle outliers, see how it affects modelling
df_model["log_juv_priors_count"] = np.log1p(df_model["juv_priors_count"])
df_model["log_adult_priors_count"] = np.log1p(df_model["adult_priors_count"])

In [ ]:
# Combine priors_count and age into frequency. Not perfect, missing dates of priors / first crime.

df_model['adult_priors_freq']=(df_model['adult_priors_count']/(df_model['age']))

df_model['adult_priors_freq_2']=((df_model['adult_priors_count']+(df_model['total_priors_count'].mean()))/((df_model['age'])+(df_model['age']).mean()))

avg_rate = df_model["adult_priors_count"].sum()/(df_model['age']).sum()
beta = 5
alfa = avg_rate * beta
df_model['adult_priors_freq_3']=((df_model['adult_priors_count']+ alfa)/((df_model['age'])+ beta))

In [66]:
df_model.to_csv(r'..\00_Data\00_Processed\df_modelo.csv', index=False)